# iTransformer — PTB-XL Membership Inference (Colab reproduction)

Notebook này giúp team **reproduce toàn bộ pipeline iTransformer** trên Google Colab (GPU) mà không cần k3s:

1. Classification + Loss / Confidence / Entropy
2. Forecasting + Loss
3. LiRA / RMIA (K=8 shadow) — cls + fcst
4. Temporal Feature — cls + fcst
5. Tổng hợp bảng AUC / TPR@0.1%FPR / TPR@1%FPR (số chính cho RQ3)

**Kết quả chuẩn đã có sẵn trong repo** (`results/itransformer/cls|fcst/*.json`, chạy trên RTX 2060).
Chỉ cần chạy lại khi muốn kiểm chứng hoặc đổi cấu hình. Checkpoint/cache nằm ở `/content/work` (không commit lên git).

> Protocol: split `ptbxl_record_s42` (seed 42, 8136 member / 500 val / 7636 non-member), điểm MIA **cao = member**.


## 0. Cấu hình

Sửa các biến dưới đây trước khi Run all:

- `TASK`: `"cls"`, `"fcst"` hoặc `"all"`
- `STAGE`: `"target"`, `"lira_rmia"`, `"temporal"` hoặc `"all"` (chạy `target` trước `lira_rmia`/`temporal`)
- `REPO_BRANCH`: branch chứa code + results (mặc định branch iTransformer hiện tại)


In [ ]:
# Runtime config / Cấu hình chạy
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

REPO_URL = "https://github.com/pearngoc/MIA.git"
REPO_BRANCH = "features/implement_itransformer_tuan"
REPO_DIR = "/content/MIA"
WORK_DIR = "/content/work"          # checkpoints + cache (not committed) / checkpoint + cache (không commit)
TASK = "all"                        # "cls" | "fcst" | "all"
STAGE = "all"                       # "target" | "lira_rmia" | "temporal" | "all"
EPOCHS, BATCH, MAX_LR, SEED, K = 50, 16, 1e-3, 42, 8

!nvidia-smi --query-gpu=name,memory.total --format=csv


## 1. Clone repo + cài dependencies

In [ ]:
# Clone or update repo / Clone hoặc cập nhật repo
import os, subprocess, sys
if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} fetch origin {REPO_BRANCH}
    !git -C {REPO_DIR} checkout {REPO_BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {REPO_BRANCH}
else:
    !git clone --branch {REPO_BRANCH} --single-branch {REPO_URL} {REPO_DIR}

# Install package + runtime deps / Cài package và thư viện chạy
!pip install -q {REPO_DIR}/experiment/itransformer wfdb scikit-learn pandas tqdm matplotlib scipy
print("repo:", REPO_DIR)


## 2. Chuẩn bị PTB-XL (không commit dataset lên git)

Chọn **một** trong hai cách:

- **Cách A (khuyến nghị cho team):** mount Google Drive đã chứa `Final_Project/dataset/archive.zip` (bản PTB-XL 1.0.1 đã nén).
- **Cách B:** tải trực tiếp từ PhysioNet: https://physionet.org/content/ptb-xl/1.0.1/ (có thể cần tài khoản PhysioNet), rồi giải nén vào `/content/ptbxl_raw`.


In [ ]:
# Option A: mount Drive if archive.zip is shared there / Cách A: mount Drive nếu đã share archive.zip
USE_DRIVE = True  # set False if using direct download / đặt False nếu tải trực tiếp
DRIVE_ROOT = "/content/drive/MyDrive/Final_Project"
PTBXL_ROOT = "/content/ptbxl_raw"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted. Expecting:", f"{DRIVE_ROOT}/dataset/archive.zip")


In [ ]:
# Extract / locate PTB-XL / Giải nén hoặc định vị PTB-XL
import glob, os, zipfile

if not glob.glob(f"{PTBXL_ROOT}/**/ptbxl_database.csv", recursive=True):
    archive = f"{DRIVE_ROOT}/dataset/archive.zip"
    assert os.path.exists(archive), (
        f"Missing {archive}. Upload PTB-XL archive.zip to Drive, "
        "or set USE_DRIVE=False and download PTB-XL 1.0.1 into /content/ptbxl_raw."
    )
    os.makedirs(PTBXL_ROOT, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        z.extractall(PTBXL_ROOT)

found = glob.glob(f"{PTBXL_ROOT}/**/ptbxl_database.csv", recursive=True)
assert found, "ptbxl_database.csv not found under /content/ptbxl_raw"
PTBXL_DIR = os.path.dirname(found[0])
print("PTBXL_DIR =", PTBXL_DIR)


## 3. Đường dẫn dùng chung (split cố định trong repo)

In [ ]:
# Shared paths / Đường dẫn dùng chung
SPLIT_PATH = f"{REPO_DIR}/dataset/splits/ptbxl_record_s42.json"
SHADOW_MASK = f"{REPO_DIR}/dataset/splits/shadow_masks_record_K8_s42.npz"
CLS_CKPT, FCST_CKPT = f"{WORK_DIR}/ckpt/itransformer_cls", f"{WORK_DIR}/ckpt/itransformer_fcst"
CLS_CACHE, FCST_CACHE = f"{WORK_DIR}/cache/cls", f"{WORK_DIR}/cache/fcst"
CLS_RESULTS, FCST_RESULTS = f"{REPO_DIR}/results/itransformer/cls", f"{REPO_DIR}/results/itransformer/fcst"
for d in [CLS_CKPT, FCST_CKPT, CLS_CACHE, FCST_CACHE, CLS_RESULTS, FCST_RESULTS]:
    os.makedirs(d, exist_ok=True)
assert os.path.exists(SPLIT_PATH) and os.path.exists(SHADOW_MASK)
print("split:", SPLIT_PATH)
print("shadow mask:", SHADOW_MASK)


## 4. Target models (Loss / Conf / Entropy)

- cls → `loss.json`, `conf.json`, `entropy.json`
- fcst → `loss.json`


In [ ]:
# Train target models / Huấn luyện model đích
import sys
run_cls = TASK in ("cls", "all")
run_fcst = TASK in ("fcst", "all")
do_target = STAGE in ("target", "all")

if do_target and run_cls:
    !{sys.executable} -m itransformer.train_cls --ptbxl-dir {PTBXL_DIR} --split-path {SPLIT_PATH} \
        --cache-dir {CLS_CACHE} --ckpt-dir {CLS_CKPT} --results-dir {CLS_RESULTS} \
        --epochs {EPOCHS} --batch {BATCH} --max-lr {MAX_LR} --seed {SEED}
if do_target and run_fcst:
    !{sys.executable} -m itransformer.train_fcst --ptbxl-dir {PTBXL_DIR} --split-path {SPLIT_PATH} \
        --cache-dir {FCST_CACHE} --ckpt-dir {FCST_CKPT} --results-dir {FCST_RESULTS} \
        --epochs {EPOCHS} --batch {BATCH} --max-lr {MAX_LR} --seed {SEED}
print("target stage done")


## 5. LiRA / RMIA (K=8 shadow)

Cần chạy mục 4 trước (cần checkpoint/logits của target ở epoch cuối).


In [ ]:
# LiRA/RMIA with K shadow models / LiRA/RMIA với K model shadow
do_shadow = STAGE in ("lira_rmia", "all")
if do_shadow and run_cls:
    !{sys.executable} -m itransformer.train_lira_rmia_cls --ptbxl-dir {PTBXL_DIR} --split-path {SPLIT_PATH} \
        --shadow-mask {SHADOW_MASK} --cache-dir {CLS_CACHE} --ckpt-dir {CLS_CKPT} --results-dir {CLS_RESULTS} \
        --epochs {EPOCHS} --batch {BATCH} --max-lr {MAX_LR} --seed {SEED} --k {K}
if do_shadow and run_fcst:
    !{sys.executable} -m itransformer.train_lira_rmia_fcst --ptbxl-dir {PTBXL_DIR} --split-path {SPLIT_PATH} \
        --shadow-mask {SHADOW_MASK} --cache-dir {FCST_CACHE} --ckpt-dir {FCST_CKPT} --results-dir {FCST_RESULTS} \
        --epochs {EPOCHS} --batch {BATCH} --max-lr {MAX_LR} --seed {SEED} --k {K}
print("lira_rmia stage done")


## 6. Temporal Feature (gray-box)

- cls: quỹ đạo CE/entropy/margin qua epoch 5/10/25/50
- fcst: DFT 8 dải + trend/drift trên residual


In [ ]:
# Temporal Feature attacks / Attack Temporal Feature
do_temporal = STAGE in ("temporal", "all")
if do_temporal and run_cls:
    !{sys.executable} -m itransformer.temporal_cls --ptbxl-dir {PTBXL_DIR} --split-path {SPLIT_PATH} \
        --cache-dir {CLS_CACHE} --ckpt-dir {CLS_CKPT} --results-dir {CLS_RESULTS} --seed {SEED}
if do_temporal and run_fcst:
    !{sys.executable} -m itransformer.temporal_fcst --ptbxl-dir {PTBXL_DIR} --split-path {SPLIT_PATH} \
        --cache-dir {FCST_CACHE} --ckpt-dir {FCST_CKPT} --results-dir {FCST_RESULTS} --seed {SEED}
print("temporal stage done")


## 7. Tổng hợp kết quả (RQ3)

Số chính: `loss`, `lira` (online), `rmia` (offline), `temporal` (+ `conf`/`entropy` cho cls).


In [ ]:
# Summarise metrics / Tổng hợp metric
import json, pathlib
rows = []
for task, folder in [("cls", CLS_RESULTS), ("fcst", FCST_RESULTS)]:
    for p in sorted(pathlib.Path(folder).glob("*.json")):
        d = json.loads(p.read_text())
        ci = (d.get("bootstrap_ci") or {}).get("tpr_at_1fpr", [None, None])
        rows.append((task, d.get("attack", p.stem), d.get("auc"),
                     d.get("tpr_at_0.1fpr"), d.get("tpr_at_1fpr"), ci))
print(f"{'task':5} {'attack':14} {'AUC':>7} {'TPR@0.1%':>9} {'TPR@1%':>8}  95%CI(TPR@1%)")
for task, attack, auc, t01, t1, ci in rows:
    print(f"{task:5} {attack:14} {auc:7.3f} {t01:9.3f} {t1:8.3f}  [{ci[0]:.3f}, {ci[1]:.3f}]")


In [ ]:
# ROC log-log for main attacks / ROC log-log cho attack chính
import json, pathlib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for ax, (task, folder, names) in zip(axes, [
    ("cls", CLS_RESULTS, ["loss", "lira", "rmia", "temporal"]),
    ("fcst", FCST_RESULTS, ["loss", "lira", "rmia", "temporal"]),
]):
    for name in names:
        p = pathlib.Path(folder) / f"{name}.json"
        if not p.exists():
            continue
        r = json.loads(p.read_text())
        sm, sn = np.array(r["scores_member"]), np.array(r["scores_nonmember"])
        fpr, tpr, _ = roc_curve(np.r_[np.ones(len(sm)), np.zeros(len(sn))], np.r_[sm, sn])
        ax.plot(fpr, tpr, label=f"{name} AUC={r['auc']:.3f}")
    ax.plot([1e-4, 1], [1e-4, 1], "k--", lw=0.8)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(1e-4, 1); ax.set_ylim(1e-4, 1)
    ax.set_xlabel("FPR"); ax.set_title(f"iTransformer {task}"); ax.legend(fontsize=8)
axes[0].set_ylabel("TPR")
plt.tight_layout(); plt.show()


## 8. Đóng gói kết quả để commit / tải về

Kết quả JSON đã ghi thẳng vào `{REPO_DIR}/results/itransformer/...`. Cell dưới tạo zip để tải về,
hoặc bạn có thể `git -C /content/MIA add results/itransformer report && git commit` rồi push bằng token của mình.
**Không** commit checkpoint (`/content/work`), dataset thô hay file zip này lên git.


In [ ]:
# Package results for download / Đóng gói kết quả để tải về
!cd {REPO_DIR} && zip -r /content/itransformer_results.zip results/itransformer report/iTransformer_RQ3.md
!git -C {REPO_DIR} status --short
print("Download: /content/itransformer_results.zip (also in Files panel)")
